# N-step SARSA: bootstrap after several rewards

N-step SARSA uses the sampled return

$$G_{t:t+n}=\sum_{k=0}^{n-1}\gamma^kR_{t+k+1}+\gamma^nQ(S_{t+n},A_{t+n}),$$
$$Q(S_t,A_t)\leftarrow Q(S_t,A_t)+\alpha\left[G_{t:t+n}-Q(S_t,A_t)\right].$$

Here $n$ is the horizon, $\gamma$ the discount factor, $\alpha$ the learning rate, and $A_{t+n}$ the action selected by the same epsilon-greedy behavior policy. The bootstrap is omitted when termination arrives first.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

ENV_ID = "FrozenLake-v1"
TOTAL_TIMESTEPS = 8_000
N_STEPS = 4
LEARNING_RATE = 0.2
GAMMA = 0.99
EPSILON_START = 1.0
EPSILON_END = 0.05
EXPLORATION_STEPS = 5_000
SEED = 7

env = gym.make(ENV_ID, is_slippery=False)
rng = np.random.default_rng(SEED)
q_table = np.zeros((env.observation_space.n, env.action_space.n))

## 1. Collect on-policy episodes

States and actions are stored together because the n-step bootstrap must use the future action actually selected by the behavior policy.

In [ ]:
def epsilon_at(step):
    fraction = min(step / EXPLORATION_STEPS, 1.0)
    return EPSILON_START + fraction * (EPSILON_END - EPSILON_START)


def select_action(state, step, deterministic=False):
    if not deterministic and rng.random() < epsilon_at(step):
        return int(rng.integers(q_table.shape[1]))
    return int(np.argmax(q_table[state]))


def collect_episode(global_step):
    states, actions, rewards = [], [], []
    state, _ = env.reset()
    action = select_action(state, global_step)
    terminated = truncated = False
    while not (terminated or truncated):
        states.append(state)
        actions.append(action)
        next_state, reward, terminated, truncated, _ = env.step(action)
        rewards.append(reward)
        if not terminated:
            next_action = select_action(next_state, global_step + len(rewards))
        state = next_state
        if not (terminated or truncated):
            action = next_action
    truncation_value = 0.0
    if truncated and not terminated:
        truncation_value = q_table[next_state, next_action]
    return states, actions, rewards, terminated, truncated, truncation_value

## 2. Build each n-step return

If at least $n$ sampled rewards remain, bootstrap from the recorded future state-action pair. True termination contributes zero. At a time-limit truncation, bootstrap from the boundary state because the underlying MDP did not terminate.

In [ ]:
def update_episode(states, actions, rewards, terminated, truncated, truncation_value):
    length = len(rewards)
    for start in range(length):
        end = min(start + N_STEPS, length)
        target = sum(
            GAMMA ** (index - start) * rewards[index]
            for index in range(start, end)
        )
        if start + N_STEPS < length:
            target += GAMMA ** N_STEPS * q_table[
                states[start + N_STEPS], actions[start + N_STEPS]
            ]
        elif truncated and not terminated:
            target += GAMMA ** (length - start) * truncation_value
        state, action = states[start], actions[start]
        q_table[state, action] += LEARNING_RATE * (target - q_table[state, action])


def train(total_timesteps):
    returns = []
    steps = 0
    while steps < total_timesteps:
        episode = collect_episode(steps)
        states, actions, rewards, terminated, truncated, truncation_value = episode
        update_episode(states, actions, rewards, terminated, truncated, truncation_value)
        steps += len(rewards)
        returns.append(sum(rewards))
    return returns


episode_returns = train(TOTAL_TIMESTEPS)
env.close()

## 3. Plot learning

The moving average estimates recent success while the raw binary returns show individual episodes.

In [ ]:
window = min(50, len(episode_returns))
moving_average = np.convolve(episode_returns, np.ones(window) / window, mode="valid")
plt.plot(episode_returns, alpha=0.25, label="Episode return")
plt.plot(np.arange(window - 1, len(episode_returns)), moving_average, label=f"{window}-episode average")
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title("n-step SARSA on FrozenLake-v1")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 4. Evaluate the greedy policy

This opens a separate rendered environment and disables exploration for 5 episodes.

In [ ]:
evaluation_env = gym.make(ENV_ID, render_mode="human", is_slippery=False)
evaluation_returns = []
try:
    for episode in range(5):
        state, _ = evaluation_env.reset(seed=10 + episode)
        total_reward = 0.0
        done = False
        while not done:
            action = select_action(state, TOTAL_TIMESTEPS, deterministic=True)
            state, reward, terminated, truncated, _ = evaluation_env.step(action)
            total_reward += reward
            done = terminated or truncated
        evaluation_returns.append(total_reward)
finally:
    evaluation_env.close()
print("Episode returns:", evaluation_returns)